<a href="https://colab.research.google.com/github/armandoordonez/AI-Engineering/blob/main/S3_notebook_previo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 3 · Notebook previo
## Tool calling, agentes y una primera conexión a MCP

**Curso:** Ciencia de datos aumentada con IA: del dato a la decisión
**Tiempo estimado:** 40 a 50 minutos · **Entrega:** este notebook ejecutado, antes de la clase

### Qué vas a hacer
1. Darle a un LLM una **herramienta** (una función Python) que puede decidir usar por sí mismo: consultar la base `tickets_soporte.db`.
2. Construir un bucle simple de **agente**: el modelo pide usar la herramienta, tú ejecutas el código real, y le devuelves el resultado hasta que responda.
3. Ejecutar 10 preguntas de referencia y medir cuántas responde bien y qué herramientas usó.
4. (Opcional, fuera de Colab) Envolver la misma herramienta como un **servidor MCP** y conectarte a él con un cliente.

En el taller de la sesión 3 construirán el agente completo sobre un servidor MCP con los datos de su propio caso.

## 0. Configuración

In [10]:
!pip install -q anthropic pandas

In [18]:
import os, sqlite3, json, time
import pandas as pd
import anthropic

def obtener_clave(nombre):
    try:
        from google.colab import userdata
        return userdata.get(nombre)
    except Exception:
        return os.environ.get(nombre) or input(f"Pega tu {nombre}: ")

cliente = anthropic.Anthropic(
    api_key=obtener_clave("ANTHROPIC_API_KEY"),
    default_headers={"anthropic-workspace-id": "wrkspc_01VGea2uNPaBW19AGPC62p2Q"}
)

MODELO = "claude-haiku-4-5"

RUTA_DB = "/content/tickets_soporte.db"
con = sqlite3.connect(RUTA_DB)

# Crear la tabla a partir del CSV si no existe en la base de datos
try:
    pd.read_sql("SELECT 1 FROM tickets LIMIT 1", con)
except Exception:
    print("Creando la tabla 'tickets' a partir del archivo CSV...")
    df_csv = pd.read_csv("/content/tickets_soporte.csv")
    df_csv.to_sql("tickets", con, if_exists="replace", index=False)

print(pd.read_sql("SELECT COUNT(*) AS total FROM tickets", con))

   total
0    243


## 1. Tool calling: dar una herramienta al modelo
El LLM **no ejecuta código**: cuando lo necesita, pide (en un formato estructurado) que tú ejecutes una función y le devuelvas el resultado. Vamos a definir una única herramienta: `consultar_sql`, que ejecuta una consulta SQL de solo lectura sobre la base de tickets.

**Por seguridad**, cualquier herramienta que ejecute código a nombre de un modelo debe limitar lo que puede hacer. Aquí solo permitimos sentencias `SELECT`.

In [19]:
ESQUEMA_TABLA = '''
Tabla: tickets
Columnas: ticket_id (texto), fecha (texto, YYYY-MM-DD), canal (texto: web, app, telefono, tienda, redes_sociales),
ciudad (texto), producto (texto), categoria (texto: envio, facturacion, calidad, soporte_tecnico, otro),
monto_cop (número o nulo), horas_resolucion (número), satisfaccion (número 1 a 5 o nulo), texto (texto libre del ticket)
'''

def consultar_sql(sql):
    """Ejecuta una consulta SQL de solo lectura sobre la tabla `tickets` y devuelve las filas como texto."""
    if not sql.strip().lower().startswith("select"):
        return "ERROR: solo se permiten consultas SELECT."
    try:
        df = pd.read_sql(sql, con)
    except Exception as e:
        return f"ERROR al ejecutar la consulta: {e}"
    if len(df) > 30:
        return df.head(30).to_string(index=False) + f"\n... ({len(df)} filas en total, se muestran 30)"
    return df.to_string(index=False)

HERRAMIENTAS = [{
    "name": "consultar_sql",
    "description": f"Ejecuta una consulta SQL de solo lectura (SELECT) sobre la base de tickets de soporte. {ESQUEMA_TABLA}",
    "input_schema": {
        "type": "object",
        "properties": {"sql": {"type": "string", "description": "Consulta SQL SELECT a ejecutar."}},
        "required": ["sql"]
    }
}]

print(consultar_sql("SELECT categoria, COUNT(*) AS n FROM tickets GROUP BY categoria ORDER BY n DESC"))

      categoria  n
          envio 84
    facturacion 58
        calidad 51
soporte_tecnico 35
           otro 15


## 2. El bucle del agente
Un agente simple repite este ciclo: **pensar → pedir una herramienta → ejecutar → observar el resultado → responder o volver a pedir otra herramienta**. Vamos a implementarlo con un máximo de pasos, para evitar bucles infinitos.

In [21]:
def ejecutar_agente(pregunta, max_pasos=5, verbose=True):
    mensajes = [{"role": "user", "content": pregunta}]
    herramientas_usadas = []
    for paso in range(max_pasos):
        r = cliente.messages.create(model=MODELO, max_tokens=800, tools=HERRAMIENTAS, messages=mensajes)
        mensajes.append({"role": "assistant", "content": r.content})
        if r.stop_reason != "tool_use":
            texto_final = "".join(b.text for b in r.content if b.type == "text")
            return texto_final, herramientas_usadas
        bloques_resultado = []
        for bloque in r.content:
            if bloque.type == "tool_use":
                herramientas_usadas.append({"herramienta": bloque.name, "entrada": bloque.input})
                if verbose:
                    print(f"  [paso {paso+1}] llama a {bloque.name}({bloque.input})")
                if bloque.name == "consultar_sql":
                    resultado = consultar_sql(bloque.input["sql"])
                else:
                    resultado = f"ERROR: herramienta desconocida {bloque.name}"
                bloques_resultado.append({"type": "tool_result", "tool_use_id": bloque.id, "content": resultado})
        mensajes.append({"role": "user", "content": bloques_resultado})
    return "(el agente no terminó en el número máximo de pasos)", herramientas_usadas

respuesta, usadas = ejecutar_agente("¿Cuál es la ciudad con más tickets de categoría calidad?")
print("\nRespuesta:", respuesta)

  [paso 1] llama a consultar_sql({'sql': "SELECT ciudad, COUNT(*) as cantidad_tickets FROM tickets WHERE categoria = 'calidad' GROUP BY ciudad ORDER BY cantidad_tickets DESC LIMIT 1"})

Respuesta: La ciudad con más tickets de categoría **calidad** es **Cartagena** con **10 tickets**.


### TU TURNO 1
Prueba el agente con una pregunta **más difícil**, que requiera combinar dos condiciones (por ejemplo, comparar dos categorías, o filtrar por ciudad y por fecha). Observa el SQL que generó y verifica manualmente si es correcto.

In [22]:
respuesta, usadas = ejecutar_agente("Tu pregunta aquí...")
print(respuesta)
print(usadas)

respuesta_1 = """
Escribe aquí qué SQL generó, si era correcto, y qué pasaría si no lo hubieras revisado.
"""

¡Hola! Soy tu asistente de análisis de tickets de soporte. Puedo ayudarte a:

- **Consultar datos de tickets** - búsquedas por fecha, canal, ciudad, producto, categoría, etc.
- **Analizar métricas** - tiempos de resolución, satisfacción de clientes, montos, etc.
- **Identificar patrones** - problemas más comunes, canales más usados, ciudades con más tickets, etc.
- **Generar reportes** - resúmenes por categoría, por canal, rendimiento, etc.

Algunas preguntas que puedo responder:
- ¿Cuántos tickets se han resuelto en los últimos 7 días?
- ¿Cuál es el tiempo promedio de resolución por categoría?
- ¿Qué canal tiene la mejor satisfacción del cliente?
- ¿Cuáles son los problemas más frecuentes?
- ¿Qué ciudad tiene más tickets?

**¿Qué información necesitas sobre los tickets de soporte?** 📊
[]


## 3. Evaluar el agente con un conjunto de referencia
Igual que en las sesiones anteriores, "parece que funciona" no basta. Vamos a ejecutar 10 preguntas con una respuesta de referencia (calculada directamente con pandas) y comparar.

In [23]:
PREGUNTAS_AGENTE = [
    ("¿Cuántos tickets hay en total?", lambda: len(pd.read_sql("SELECT * FROM tickets", con))),
    ("¿Cuántos tickets son de categoría envio?", lambda: len(pd.read_sql("SELECT * FROM tickets WHERE categoria='envio'", con))),
    ("¿Cuál es el monto promedio de los tickets de categoria calidad (sin contar los nulos)?",
     lambda: round(pd.read_sql("SELECT monto_cop FROM tickets WHERE categoria='calidad'", con)["monto_cop"].mean(), 0)),
    ("¿Cuántas ciudades distintas aparecen en la tabla?", lambda: pd.read_sql("SELECT DISTINCT ciudad FROM tickets", con)["ciudad"].nunique()),
    ("¿Cuál es el canal con más tickets?", lambda: pd.read_sql("SELECT canal, COUNT(*) c FROM tickets GROUP BY canal ORDER BY c DESC LIMIT 1", con)["canal"][0]),
    ("¿Cuántos tickets no tienen calificación de satisfacción?", lambda: pd.read_sql("SELECT * FROM tickets WHERE satisfaccion IS NULL", con).shape[0]),
    ("¿Cuál es la horas_resolucion máxima registrada?", lambda: pd.read_sql("SELECT MAX(horas_resolucion) m FROM tickets", con)["m"][0]),
    ("¿Cuántos tickets de categoria facturacion tienen monto mayor a 100000?",
     lambda: pd.read_sql("SELECT * FROM tickets WHERE categoria='facturacion' AND monto_cop>100000", con).shape[0]),
    ("¿Cuál es la categoría con menor cantidad de tickets?", lambda: pd.read_sql("SELECT categoria, COUNT(*) c FROM tickets GROUP BY categoria ORDER BY c ASC LIMIT 1", con)["categoria"][0]),
    ("¿Cuántos tickets hay del canal app en la ciudad de Cali?",
     lambda: pd.read_sql("SELECT * FROM tickets WHERE canal='app' AND ciudad='Cali'", con).shape[0]),
]

filas = []
for pregunta, fn_referencia in PREGUNTAS_AGENTE:
    referencia = fn_referencia()
    respuesta, usadas = ejecutar_agente(pregunta, verbose=False)
    filas.append({"pregunta": pregunta, "referencia": referencia, "respuesta_agente": respuesta,
                  "num_llamadas_sql": sum(1 for u in usadas if u["herramienta"] == "consultar_sql")})
    time.sleep(0.5)

df_eval = pd.DataFrame(filas)
df_eval

,pregunta,referencia,respuesta_agente,num_llamadas_sql
0,¿Cuántos tickets hay en total?,243,Hay un total de **243 tickets** en la base de ...,1
1,¿Cuántos tickets son de categoría envio?,84,Hay **84 tickets** de categoría envío.,1
2,¿Cuál es el monto promedio de los tickets de c...,1216905.0,El monto promedio de los tickets de categoría ...,1
3,¿Cuántas ciudades distintas aparecen en la tabla?,16,En la tabla de tickets aparecen **16 ciudades ...,1
4,¿Cuál es el canal con más tickets?,redes_sociales,El canal con más tickets es **redes_sociales**...,1
5,¿Cuántos tickets no tienen calificación de sat...,32,Hay **32 tickets** que no tienen calificación ...,1
6,¿Cuál es la horas_resolucion máxima registrada?,999.0,La **horas_resolucion máxima registrada es 999...,1
7,¿Cuántos tickets de categoria facturacion tien...,5,"Hay **5 tickets** de categoría ""facturación"" q...",1
8,¿Cuál es la categoría con menor cantidad de ti...,otro,La categoría con menor cantidad de tickets es ...,1
9,¿Cuántos tickets hay del canal app en la ciuda...,7,Hay **7 tickets** del canal app en la ciudad d...,1


### TU TURNO 2
El agente responde en texto libre, así que comparar automáticamente si "acertó" no es trivial (a diferencia del JSON de la sesión 1). Revisa manualmente cada fila de `df_eval`:
1. Marca cuáles respuestas coinciden con la referencia (puedes agregar una columna `correcto` a mano o con un LLM como juez).
2. ¿En qué tipo de pregunta el agente necesitó más de una llamada a `consultar_sql`? ¿Por qué crees que pasó?

In [24]:
df_eval["correcto"] = None  # TU TURNO: complétala con True/False revisando cada fila
df_eval

respuesta_2 = """
Escribe aquí tu análisis.
"""

## 4. (Opcional, fuera de Colab) La misma herramienta como servidor MCP
**MCP (Model Context Protocol)** estandariza cómo un modelo descubre y usa herramientas: en lugar de definir `HERRAMIENTAS` a mano en cada notebook, un **servidor MCP** las expone, y cualquier cliente compatible (un IDE, un agente, otro notebook) se conecta a él sin reescribir nada. Es la misma idea de la sección 1, empaquetada de forma reutilizable.

Esto **no funciona dentro de Colab** (Colab no permite abrir un proceso servidor persistente de forma sencilla). Si quieres probarlo, hazlo en tu computador con Jupyter o VS Code:

```bash
pip install mcp
```

```python
# archivo: servidor_tickets.py
from mcp.server.fastmcp import FastMCP
import sqlite3, pandas as pd

mcp = FastMCP("tickets-tiendatech")
con = sqlite3.connect("tickets_soporte.db")

@mcp.tool()
def consultar_sql(sql: str) -> str:
    """Ejecuta una consulta SELECT de solo lectura sobre la tabla tickets."""
    if not sql.strip().lower().startswith("select"):
        return "ERROR: solo se permiten consultas SELECT."
    return pd.read_sql(sql, con).head(30).to_string(index=False)

if __name__ == "__main__":
    mcp.run()
```

Ese mismo servidor se puede conectar desde un cliente MCP (por ejemplo, Claude Desktop o Claude Code) editando su configuración para apuntar a `servidor_tickets.py`, sin volver a escribir la definición de la herramienta en cada cliente. En la sesión 3 el profesor entrega un servidor MCP ya armado para que el taller no dependa de esta instalación local.

In [25]:
# Nada que ejecutar aquí: es material de lectura para quien quiera probarlo localmente.

## 5. Cierre
### TU TURNO 3 · Reflexión final (máx. 5 líneas)
¿Qué riesgos ves en dejar que un agente ejecute SQL generado por un LLM sin que una persona lo revise? Piensa en un caso donde eso pueda salir mal, más allá de la exactitud del resultado.

In [26]:
respuesta_3 = """
Escribe aquí tu reflexión.
"""

DECLARACION_IAG = """
Nivel de uso de IAG: (1 a 5, según el acuerdo del curso)
Herramientas usadas y para qué:
Cómo verifiqué la calidad y veracidad del contenido generado:
"""

### Antes de entregar
- Sube `tickets_soporte.db` a tu sesión de Colab antes de ejecutar (se genera a partir de `tickets_soporte.csv` con `pandas.to_sql`; si no lo tienes, créalo con una celda adicional).
- Ejecuta **Entorno de ejecución → Reiniciar y ejecutar todo** y verifica que no haya errores.
- Comprueba que las celdas `respuesta_1` a `respuesta_3`, `df_eval["correcto"]` y `DECLARACION_IAG` estén completas.
- **No compartas tu clave de API.**